# Fine-tune ModernFinBERT

Fine-tune `tabularisai/ModernFinBERT` on your own labeled headlines.
Runs on free T4 GPU in ~4 minutes for 763 headlines.

## Step 1: Set HF token + install deps

Go to https://huggingface.co/settings/tokens, create a **read** token, paste it below.

In [ ]:
import os
os.environ['HF_TOKEN'] = ''  # ← paste your free HF read token here
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'  # enables fast Rust downloader

In [ ]:
!pip install -q hf-transfer transformers datasets accelerate evaluate scikit-learn

## Step 2: Setup output directory

In [ ]:
import os
OUTPUT_DIR = '/content/fine_tuned_finbert'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Model will be saved to: {OUTPUT_DIR}')

## Step 3: Upload your labeled CSV

Click **Choose Files** and select your `labeled_headlines.csv`.

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print(f'Loaded {len(df)} labeled headlines')
print(f'Distribution:\n{df["label"].value_counts().sort_index()}')

## Step 4: Load ModernFinBERT

Label mapping: 0=bearish, 1=neutral, 2=bullish

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset
from sklearn.model_selection import train_test_split

MODEL_NAME = 'tabularisai/ModernFinBERT'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: 'bearish', 1: 'neutral', 2: 'bullish'},
    label2id={'bearish': 0, 'neutral': 1, 'bullish': 2},
)
print(f'Model loaded: {model.num_parameters():,} params')

## Step 5: Tokenize and split

In [ ]:
def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(),
    test_size=0.15, stratify=df['label'], random_state=42,
)

train_ds = Dataset.from_dict({'text': train_texts, 'label': train_labels})
val_ds = Dataset.from_dict({'text': val_texts, 'label': val_labels})
train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f'Train: {len(train_ds)} samples')
print(f'Val:   {len(val_ds)} samples')

## Step 6: Train (~4 min on T4)

In [ ]:
training_args = TrainingArguments(
    output_dir='/content/checkpoints',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    fp16=True,
    report_to='none',
    save_total_limit=1,
)

import numpy as np
import evaluate
accuracy_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## Step 7: Evaluate

In [ ]:
metrics = trainer.evaluate()
print(f'Validation accuracy: {metrics["eval_accuracy"]:.3f}')
print(f'Validation loss: {metrics["eval_loss"]:.4f}')

## Step 8: Save model + tokenizer

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Clean up checkpoints to reduce ZIP size
import shutil
for d in ['/content/checkpoints']:
    if os.path.exists(d):
        shutil.rmtree(d, ignore_errors=True)

!ls -lh "{OUTPUT_DIR}"
print(f'\nModel saved to {OUTPUT_DIR}')

## Step 9: Verify + Download

Verifies model integrity, then prompts download.

In [ ]:
# Verify model loads correctly
from safetensors import safe_open
with safe_open(os.path.join(OUTPUT_DIR, 'model.safetensors'), framework='pt', device='cpu') as f:
    keys = f.keys()
print(f'Model file verified: {len(keys)} tensors, {os.path.getsize(os.path.join(OUTPUT_DIR, "model.safetensors"))/1e6:.0f} MB')
print('File integrity: OK')

In [ ]:
import shutil
import os

zip_path = '/content/fine_tuned_finbert.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive('/content/fine_tuned_finbert', 'zip', OUTPUT_DIR)

size_mb = os.path.getsize(zip_path) / 1e6
print(f'ZIP created: {zip_path} ({size_mb:.1f} MB)')

from google.colab import files
files.download(zip_path)

---
**Done!** After download, unzip to `models/fine_tuned/` in the project root.